# 07 - Identify CJEU Cases Citing Other CJEU Cases

This notebook checks which CJEU documents cite other CJEU cases from the master list.

**Approach:**
1. Load `cjeu_cases.csv` and derive `case_number` from `celex_id`
2. For each CJEU case, fetch the full document via CELEX resource endpoint (HTML EN, then HTML DE, then Branch Notice as fallback)
3. Search the document text for CJEU case references using a generic regex
4. Match found citations against the known CJEU case list
5. Export matches to `data/processed/cjeu_cjeu_case_matches.csv`

## 1. Imports and Configuration

In [1]:
import re
import time
import unicodedata
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

# Paths
DATA_DIR        = Path("data/processed")
CJEU_CASES_PATH = DATA_DIR / "cjeu_cases.csv"
OUTPUT_PATH     = DATA_DIR / "cjeu_cjeu_case_matches.csv"

# HTTP settings
REQUEST_TIMEOUT = 30   # seconds per request
REQUEST_DELAY   = 1.0  # seconds between requests (be polite)
MAX_RETRIES     = 2

# Context window around each match (characters)
CONTEXT_CHARS = 200

print("Configuration loaded.")

Configuration loaded.


## 2. Derive `case_number` from `celex_id`

In [2]:
# CELEX format for CJEU cases: 6{year}{court}{number}
# Examples:
#   62012CJ0348 -> C-348/12
#   62012TJ0234 -> T-234/12
#   61993CJ0310 -> C-310/93

_COURT_MAP = {"CJ": "C", "TJ": "T", "FJ": "F"}

_CELEX_RE = re.compile(
    r"^6"
    r"(?P<year>\d{4})"
    r"(?P<court>CJ|TJ|FJ)"
    r"(?P<number>\d+)"
    r"$",
    re.IGNORECASE,
)


def derive_case_number_from_celex(celex_id: str) -> str:
    """
    Derive a normalised CJEU case number from a CELEX ID.

    Examples:
      '62012CJ0348' -> 'C-348/12'
      '62012TJ0234' -> 'T-234/12'
      '61993CJ0310' -> 'C-310/93'

    Returns an empty string if the CELEX ID does not match the expected pattern.
    """
    m = _CELEX_RE.match(celex_id.strip())
    if not m:
        return ""
    court  = _COURT_MAP.get(m.group("court").upper(), m.group("court").upper())
    number = str(int(m.group("number")))   # strip leading zeros
    year   = m.group("year")[-2:]           # last two digits
    return f"{court}-{number}/{year}"


# Smoke-tests
print("=== derive_case_number_from_celex ===")
for celex, expected in [
    ("62012CJ0348", "C-348/12"),
    ("62012TJ0234", "T-234/12"),
    ("61993CJ0310", "C-310/93"),
    ("62009FJ0012", "F-12/09"),
    ("62023CJ0051", "C-51/23"),
]:
    result = derive_case_number_from_celex(celex)
    status = "OK" if result == expected else f"FAIL (got {result!r})"
    print(f"  [{status}] {celex} -> {result!r}")

=== derive_case_number_from_celex ===
  [OK] 62012CJ0348 -> 'C-348/12'
  [OK] 62012TJ0234 -> 'T-234/12'
  [OK] 61993CJ0310 -> 'C-310/93'
  [OK] 62009FJ0012 -> 'F-12/09'
  [OK] 62023CJ0051 -> 'C-51/23'


## 3. URL Helper Functions

In [3]:
def build_celex_html_url(celex_id: str) -> str:
    """Build the official CELEX resource HTML URL."""
    return f"https://publications.europa.eu/resource/celex/{celex_id}"


def build_cellar_branch_url(cellar_id: str) -> str:
    """Build the CELLAR branch notice URL."""
    return f"https://publications.europa.eu/resource/cellar/{cellar_id}?language=eng"


print(build_celex_html_url("62023CJ0051"))
print(build_cellar_branch_url("ba1070bd-c40d-4171-88df-8578a05e9d17"))

https://publications.europa.eu/resource/celex/62023CJ0051
https://publications.europa.eu/resource/cellar/ba1070bd-c40d-4171-88df-8578a05e9d17?language=eng


## 4. Document Fetching Functions

In [4]:
def _get_with_retry(url: str, headers: dict) -> requests.Response | None:
    """GET a URL with simple retry logic. Returns None on failure."""
    for attempt in range(MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 200:
                return resp
            if resp.status_code in (404, 403, 410):
                return None
        except requests.RequestException:
            pass
        if attempt < MAX_RETRIES:
            time.sleep(REQUEST_DELAY * 2)
    return None


def normalize_text(raw: str) -> str:
    """Normalize Unicode, unify dashes, collapse whitespace."""
    text = unicodedata.normalize("NFKC", raw)
    text = re.sub(r"[‐-―−]", "-", text)
    text = re.sub(r"[ 	]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_text_from_html(html_bytes: bytes) -> str:
    """Parse HTML, strip scripts/styles, return visible text."""
    soup = BeautifulSoup(html_bytes, "html.parser")
    for tag in soup(["script", "style", "head", "meta", "link"]):
        tag.decompose()
    raw = soup.get_text(separator=" ")
    return normalize_text(raw)


def fetch_html_by_celex(celex_id: str, language: str = "eng") -> tuple[str, str]:
    """
    Fetch HTML document via the official CELEX resource endpoint.

    Returns (text, source_url). text is empty string on failure.
    """
    url = build_celex_html_url(celex_id)
    headers = {
        "User-Agent": "Mozilla/5.0 (research; masterarbeit) compatible",
        "Accept": "text/html",
        "Accept-Language": language,
    }
    resp = _get_with_retry(url, headers)
    if resp is not None:
        content_type = resp.headers.get("Content-Type", "")
        if "html" in content_type or "text" in content_type:
            text = extract_text_from_html(resp.content)
            if len(text) > 200:
                return text, url
    return "", url


def fetch_branch_notice_by_cellar(cellar_id: str) -> tuple[str, str]:
    """
    Fetch Branch Notice XML via the CELLAR endpoint.

    Returns (text, source_url). text is empty string on failure.
    """
    url = build_cellar_branch_url(cellar_id)
    headers = {
        "User-Agent": "Mozilla/5.0 (research; masterarbeit) compatible",
        "Accept": "application/xml;notice=branch",
    }
    resp = _get_with_retry(url, headers)
    if resp is not None:
        text = normalize_text(resp.text)
        if len(text) > 200:
            return text, url
    return "", url


def fetch_document_text(celex_id: str, cellar_id: str) -> tuple[str, str, str]:
    """
    Fetch the full text of a CJEU document.

    Priority:
      1. HTML via resource/celex/{celex_id} - English
      2. HTML via resource/celex/{celex_id} - German
      3. Branch Notice via resource/cellar/{cellar_id} (optional fallback)

    Returns (text, source_url, document_format). text is empty string on failure.
    """
    time.sleep(REQUEST_DELAY)

    # 1. HTML English
    text, url = fetch_html_by_celex(celex_id, language="eng")
    if text:
        return text, url, "html_eng"

    # 2. HTML German
    text, url = fetch_html_by_celex(celex_id, language="deu")
    if text:
        return text, url, "html_deu"

    # 3. Branch Notice via CELLAR (optional fallback)
    if cellar_id:
        text, url = fetch_branch_notice_by_cellar(cellar_id)
        if text:
            return text, url, "branch_notice"

    return "", "", "none"


print("Document fetching functions defined.")

Document fetching functions defined.


## 5. Generic Regex for CJEU Case Citations

In [5]:
# Generic pattern that matches CJEU case numbers in text.
# Handles:
#   - Court prefixes: C, T, F
#   - Various dashes: -, en-dash, and other Unicode hyphens
#   - Optional spaces around the dash
#   - Optional suffixes: P, R, PPU, RX, etc.
#   - Optional leading keywords: "Case", "Joined Cases", "Cases"

_CJEU_CASE_RE = re.compile(
    r"""
    (?:(?:Joined\s+Cases?|Cases?)\s+)?   # optional prefix keyword
    (?P<court>[CTF])                      # court letter
    \s*[-\u2010-\u2015\u2212]\s*          # dash (various forms), optional spaces
    (?P<number>\d+)                       # case number
    /                                     # slash separator
    (?P<year>\d{2,4})                     # year (2 or 4 digits)
    (?:\s+(?P<suffix>[A-Z]{1,4}))?        # optional suffix: P, R, PPU, RX, ...
    """,
    re.VERBOSE | re.IGNORECASE,
)


def normalize_cjeu_case_number(case_str: str) -> str:
    """
    Normalise a raw CJEU case string to a canonical form.

    Examples:
      'C - 348/12'   -> 'C-348/12'
      'T-234/12'     -> 'T-234/12'
      'C-123/04 P'   -> 'C-123/04 P'
      'Case C-51/23' -> 'C-51/23'
    """
    m = _CJEU_CASE_RE.search(case_str)
    if not m:
        return case_str.strip()
    court  = m.group("court").upper()
    number = m.group("number")
    year   = m.group("year")
    suffix = m.group("suffix")
    base   = f"{court}-{number}/{year}"
    return f"{base} {suffix.upper()}" if suffix else base


def extract_cjeu_case_citations(text: str) -> list[dict]:
    """
    Extract all CJEU case citations from a text.

    Returns a list of dicts with keys:
      matched_text  - the raw matched string
      normalized    - the normalised case number (e.g. 'C-348/12')
      start         - match start position in text
      end           - match end position in text
    """
    results = []
    for m in _CJEU_CASE_RE.finditer(text):
        raw        = m.group(0)
        normalized = normalize_cjeu_case_number(raw)
        results.append({
            "matched_text": raw,
            "normalized":   normalized,
            "start":        m.start(),
            "end":          m.end(),
        })
    return results


# Smoke-tests
print("=== normalize_cjeu_case_number ===")
for raw, expected in [
    ("C-348/12",     "C-348/12"),
    ("T-234/12",     "T-234/12"),
    ("F-12/09",      "F-12/09"),
    ("C - 348/12",   "C-348/12"),
    ("C-123/04 P",   "C-123/04 P"),
    ("Case C-51/23", "C-51/23"),
]:
    result = normalize_cjeu_case_number(raw)
    status = "OK" if result == expected else f"FAIL (got {result!r})"
    print(f"  [{status}] {raw!r} -> {result!r}")

print()
print("=== extract_cjeu_case_citations ===")
sample = "See Case C-348/12 and Joined Cases T-1/20 and T-2/20, also F-12/09 P."
for hit in extract_cjeu_case_citations(sample):
    print(f"  {hit['matched_text']!r:30} -> {hit['normalized']!r}")

=== normalize_cjeu_case_number ===
  [OK] 'C-348/12' -> 'C-348/12'
  [OK] 'T-234/12' -> 'T-234/12'
  [OK] 'F-12/09' -> 'F-12/09'
  [OK] 'C - 348/12' -> 'C-348/12'
  [OK] 'C-123/04 P' -> 'C-123/04 P'
  [OK] 'Case C-51/23' -> 'C-51/23'

=== extract_cjeu_case_citations ===
  'Case C-348/12 and'            -> 'C-348/12 AND'
  'Joined Cases T-1/20 and'      -> 'T-1/20 AND'
  'T-2/20'                       -> 'T-2/20'
  'F-12/09 P'                    -> 'F-12/09 P'


## 6. Load CJEU Cases and Derive `case_number`

In [6]:
cjeu_cases = pd.read_csv(CJEU_CASES_PATH, dtype=str).fillna("")

print(f"CJEU cases loaded: {len(cjeu_cases):,} rows")
print("Columns:", list(cjeu_cases.columns))

CJEU cases loaded: 1,354 rows
Columns: ['cellar_id', 'celex_id', 'title', 'document_date', 'court_level', 'resource_type_code', 'treaty_code', 'ct_codes', 'case_law_parties_raw']


In [7]:
cjeu_cases["case_number"] = cjeu_cases["celex_id"].apply(derive_case_number_from_celex)

derived_count = (cjeu_cases["case_number"] != "").sum()
print(f"case_number derived for {derived_count:,} / {len(cjeu_cases):,} rows")
cjeu_cases[["celex_id", "case_number"]].head(10)

case_number derived for 1,100 / 1,354 rows


,celex_id,case_number
0,61989TJ0068,T-68/89
1,61994CJ0068,C-68/94
2,62002TJ0038,T-38/02
3,61989TJ0065,T-65/89
4,61976CJ0085,C-85/76
5,61993CJ0310,C-310/93
6,61994CJ0333,C-333/94
7,62006TJ0170,T-170/06
8,61999TJ0219,T-219/99
9,62004TJ0464,T-464/04


In [8]:
# Build a lookup dict: case_number -> row  (for fast matching)
cjeu_lookup = (
    cjeu_cases[cjeu_cases["case_number"] != ""]
    .set_index("case_number")
    .to_dict(orient="index")
)

print(f"CJEU lookup entries: {len(cjeu_lookup):,}")

CJEU lookup entries: 1,100


## 7. Match Extraction Function

In [9]:
def find_cjeu_matches_in_text(
    text: str,
    source_row: pd.Series,
    cjeu_lookup: dict,
    source_url: str,
    doc_format: str,
) -> list[dict]:
    """
    Search text for CJEU case citations and match them against the known CJEU list.

    - Skips self-references (source case citing itself).
    - Deduplicates: same source + same normalised citation -> keep first match only.

    Returns a list of match record dicts.
    """
    source_celex_id    = str(source_row.get("celex_id", "")).strip()
    source_cellar_id   = str(source_row.get("cellar_id", "")).strip()
    source_case_number = str(source_row.get("case_number", "")).strip()
    source_title       = str(source_row.get("title", "")).strip()
    source_date        = str(source_row.get("document_date", "")).strip()

    citations = extract_cjeu_case_citations(text)

    records = []
    seen    = set()  # (source_celex_id, normalized_match)

    for citation in citations:
        normalized = citation["normalized"]

        # Skip self-references
        if normalized == source_case_number:
            continue

        # Check if citation matches a known CJEU case
        if normalized not in cjeu_lookup:
            continue

        dedup_key = (source_celex_id, normalized)
        if dedup_key in seen:
            continue
        seen.add(dedup_key)

        target  = cjeu_lookup[normalized]
        start   = max(0, citation["start"] - CONTEXT_CHARS)
        end     = min(len(text), citation["end"] + CONTEXT_CHARS)
        context = text[start:end].replace("\n", " ")

        records.append({
            "source_celex_id":      source_celex_id,
            "source_cellar_id":     source_cellar_id,
            "source_case_number":   source_case_number,
            "source_title":         source_title,
            "source_document_date": source_date,
            "target_celex_id":      target.get("celex_id", ""),
            "target_cellar_id":     target.get("cellar_id", ""),
            "target_case_number":   normalized,
            "target_title":         target.get("title", ""),
            "matched_text":         citation["matched_text"],
            "normalized_match":     normalized,
            "match_context":        context,
            "document_source_url":  source_url,
            "document_format":      doc_format,
            "processing_status":    "matched",
        })

    return records


print("Match extraction function defined.")

Match extraction function defined.


## 8. Main Processing Loop

In [10]:
all_matches = []
total = len(cjeu_cases)

for i, (idx, source_row) in enumerate(cjeu_cases.iterrows(), start=1):
    celex_id  = str(source_row.get("celex_id", "")).strip()
    cellar_id = str(source_row.get("cellar_id", "")).strip()

    print(f"[{i}/{total}] {celex_id}", end="", flush=True)

    try:
        text, source_url, doc_format = fetch_document_text(celex_id, cellar_id)
    except Exception as e:
        print(f" -> fetch_error: {e}")
        continue

    if not text:
        print(f" -> fetch_failed [{doc_format}]")
        continue

    matches = find_cjeu_matches_in_text(text, source_row, cjeu_lookup, source_url, doc_format)
    all_matches.extend(matches)
    print(f" -> {len(matches)} match(es) [{doc_format}]")

print(f"\nDone. Total matches found: {len(all_matches):,}")

[1/1354] 61989TJ0068 -> 1 match(es) [html_eng]
[2/1354] 61994CJ0068 -> 1 match(es) [html_eng]
[3/1354] 62002TJ0038 -> 5 match(es) [html_eng]
[4/1354] 61989TJ0065 -> 0 match(es) [html_eng]
[5/1354] 61976CJ0085 -> 0 match(es) [html_eng]
[6/1354] 61993CJ0310 -> 0 match(es) [html_eng]
[7/1354] 61994CJ0333 -> 0 match(es) [html_eng]
[8/1354] 62006TJ0170 -> 1 match(es) [html_eng]
[9/1354] 61999TJ0219 -> 2 match(es) [html_eng]
[10/1354] 62004TJ0464 -> 1 match(es) [html_eng]
[11/1354] 62001CJ0082 -> 0 match(es) [html_eng]
[12/1354] 61983CJ0298 -> 0 match(es) [html_eng]
[13/1354] 62001TJ0057 -> 3 match(es) [html_eng]
[14/1354] 61998CO0428 -> 1 match(es) [html_eng]
[15/1354] 61991TJ0032 -> 1 match(es) [html_eng]
[16/1354] 62000CO0241 -> 0 match(es) [html_eng]
[17/1354] 61999TJ0009 -> 0 match(es) [html_eng]
[18/1354] 61986CO0062 -> 0 match(es) [html_eng]
[19/1354] 61997CJ0007 -> 0 match(es) [html_eng]
[20/1354] 61991TJ0083 -> 1 match(es) [html_eng]
[21/1354] 61997TJ0228 -> 5 match(es) [html_eng]
[

## 9. Build Results DataFrame and Export

In [11]:
OUTPUT_COLUMNS = [
    "source_celex_id",
    "source_cellar_id",
    "source_case_number",
    "source_title",
    "source_document_date",
    "target_celex_id",
    "target_cellar_id",
    "target_case_number",
    "target_title",
    "matched_text",
    "normalized_match",
    "match_context",
    "document_source_url",
    "document_format",
    "processing_status",
]

if all_matches:
    results_df = pd.DataFrame(all_matches, columns=OUTPUT_COLUMNS)
else:
    results_df = pd.DataFrame(columns=OUTPUT_COLUMNS)

print(f"Result rows: {len(results_df):,}")
results_df.head()

Result rows: 1,154


,source_celex_id,source_cellar_id,source_case_number,source_title,source_document_date,target_celex_id,target_cellar_id,target_case_number,target_title,matched_text,normalized_match,match_context,document_source_url,document_format,processing_status
0,61989TJ0068,ba2d7353-f3e9-4826-b04a-cc79d6cdb8d4,T-68/89,,1992-03-10,61989TJ0069,9b37fa92-cdfb-4245-aa42-bdabd8726434,T-69/89,,Cases T-69/89,T-69/89,by the Court were heard at the hearing which ...,https://publications.europa.eu/resource/celex/...,html_eng,matched
1,61994CJ0068,ba1070bd-c40d-4171-88df-8578a05e9d17,C-68/94,,1998-03-31,61985CJ0089,5ed612ae-790c-4091-bf05-0a6d7cbd9142,C-89/85,,Joined Cases C-89/85,C-89/85,ithin the meaning of Article 8(2) of the Regul...,https://publications.europa.eu/resource/celex/...,html_eng,matched
2,62002TJ0038,b6ccae39-9aa5-4253-953a-22b37892e1ff,T-38/02,,2005-10-25,61995TJ0025,cbe23c9f-8720-43fd-805f-b982c4fca32d,T-25/95,,Joined Cases T-25/95,T-25/95,t have been sufficient to prove the alleged pr...,https://publications.europa.eu/resource/celex/...,html_eng,matched
3,62002TJ0038,b6ccae39-9aa5-4253-953a-22b37892e1ff,T-38/02,,2005-10-25,61989TJ0068,ba2d7353-f3e9-4826-b04a-cc79d6cdb8d4,T-68/89,,Joined Cases T-68/89,T-68/89,ria used in determining the amount of the fine...,https://publications.europa.eu/resource/celex/...,html_eng,matched
4,62002TJ0038,b6ccae39-9aa5-4253-953a-22b37892e1ff,T-38/02,,2005-10-25,61994TJ0374,54c542f8-acf0-4235-a4ac-c25873562f29,T-374/94,,Joined Cases T-374/94,T-374/94,ings or concerted practice at issue is liable ...,https://publications.europa.eu/resource/celex/...,html_eng,matched


In [12]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"Saved {len(results_df):,} rows to: {OUTPUT_PATH}")

Saved 1,154 rows to: data\processed\cjeu_cjeu_case_matches.csv


## 10. Summary Statistics

In [13]:
if not results_df.empty:
    print("=== Document Format Distribution ===")
    print(results_df["document_format"].value_counts().to_string())
    print()
    print("=== Top 10 Most-Cited CJEU Cases ===")
    print(
        results_df.groupby(["target_case_number", "target_title"])
        .size()
        .sort_values(ascending=False)
        .head(10)
        .to_string()
    )
    print()
    print("=== CJEU Documents with Most CJEU Citations ===")
    print(
        results_df.groupby("source_celex_id")["target_case_number"]
        .nunique()
        .sort_values(ascending=False)
        .head(10)
        .to_string()
    )
else:
    print("No matches found.")

=== Document Format Distribution ===
document_format
html_eng    1129
html_deu      25

=== Top 10 Most-Cited CJEU Cases ===
target_case_number  target_title
C-89/85                             56
T-25/95                             51
T-79/89                             32
T-374/94                            27
T-236/01                            27
C-278/92                            25
C-329/93                            25
T-304/94                            25
T-337/94                            25
T-311/94                            24

=== CJEU Documents with Most CJEU Citations ===
source_celex_id
61994TJ0334    18
61994TJ0352    17
61994TJ0317    16
61994TJ0311    16
61994TJ0339    16
61994TJ0348    16
61994TJ0337    16
61994TJ0310    16
62002TJ0048    15
61994TJ0338    15
